# GRADE --- RoBERTa, all 15 folds

The main sweep (`GRADE_full_study_v2.ipynb`) trains DeBERTa-v3-small on
all 15 leave-one-assignment-out folds. A first check on the two extreme
folds found the architectures agree on the easy fold (Car-free cities,
both ~99.6%) but disagree sharply on the hard one: DeBERTa fell to 93.47%
and lost to the word-frequency control by 5.26 points on Exploring Venus,
while RoBERTa reached 99.84% and beat the control there.

One hard fold is not enough to know whether that is a pattern or a
one-off. This notebook runs RoBERTa-base on the remaining 13 folds so the
full 15-fold comparison can be made, and the paper's "One architecture"
limitation can be answered properly rather than from two folds.

It reuses the corpus, LOTO splits and control comparisons already on disk
under `data/loto/` and `results/loto_baselines.csv` --- architecture has
no bearing on any of those, so nothing is rebuilt. It writes into the same
`results/loto_runs.csv` and `results/loto_predictions/` the main sweep
uses, tagged `model='roberta'`. The two folds already run (13 and 1) are
skipped automatically.

## 0. Environment

In [1]:
import os, sys, gc, time, random, hashlib, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')

import numpy as np
import pandas as pd

REPO    = os.path.abspath('..')
DATA    = os.path.join(REPO, 'data')
LOTO    = os.path.join(DATA, 'loto')
DATASETS = os.path.join(REPO, 'datasets')
RESULTS = os.path.join(REPO, 'results')
PREDS   = os.path.join(RESULTS, 'loto_predictions')
CKPT    = os.path.join(REPO, 'models', 'loto')
for d in (PREDS, CKPT):
    os.makedirs(d, exist_ok=True)

RUNS     = os.path.join(RESULTS, 'loto_runs.csv')
BASE     = os.path.join(RESULTS, 'loto_baselines.csv')
PERSUADE = os.path.join(DATASETS, 'Persuade 2.0',
                        'persuade_2.0_human_scores_demo_id_github.csv')
FOLDS    = os.path.join(LOTO, 'folds.csv')

def h(s):
    return hashlib.md5(str(s).strip().lower().encode('utf8', 'ignore')).hexdigest()

import torch
print('torch          ', torch.__version__)
print('cuda available ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device         ', torch.cuda.get_device_name(0))
    print('vram           ', '%.1f GB' %
          (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('\n*** NO GPU - this will be impractically slow. ***')
import transformers
print('transformers   ', transformers.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

folds = pd.read_csv(FOLDS)
print('\nfolds on disk:', len(folds))
print(folds[['fold', 'held_out_topic', 'n_train', 'n_val', 'n_test']].to_string(index=False))

torch           2.5.1+cu118
cuda available  True
device          NVIDIA GeForce RTX 3060 Ti
vram            8.6 GB
transformers    4.57.6

folds on disk: 15
 fold                        held_out_topic  n_train  n_val  n_test
    0                     Distance learning    23966   4230    4314
    1                       Car-free cities    26252   4632    4102
    2      Does the electoral college work?    26814   4732    3440
    3             Seeking multiple opinions    23578   4160    3104
    4  Mandatory extracurricular activities    27348   4826    2812
    5                       Summer projects    28122   4962    1902
    6           Facial action coding system    28180   4974    1834
    7                     Community service    28804   5084    1100
    8         "A Cowboy Who Rode the Waves"    28850   5092    1046
    9 Grades for extracurricular activities    28906   5102     980
   10                 Cell phones at school    28952   5110     926
   11                    Ph

## 1. Configuration

In [2]:
STAGE = 1                     # 1 = all 15 folds at seed 42; 2 = extremes at extra seeds
MODEL = 'roberta'
HF_ID = 'roberta-base'
EXTRA_SEEDS = [123, 2024]     # only used at STAGE 2, on the best/worst folds

CONFIG = {'max_len': 256, 'lr': 2e-5, 'batch_size': 8, 'epochs': 3}

print('detector %s -> %s' % (MODEL, HF_ID))
print('config   %s' % CONFIG)
print('stage    %d' % STAGE)

detector roberta -> roberta-base
config   {'max_len': 256, 'lr': 2e-05, 'batch_size': 8, 'epochs': 3}
stage    1


## 2. Training and evaluation core

Identical to the main notebook's section 4 core: same dataset class, same
encode/predict/metrics functions, same AdamW recipe, same best-checkpoint
selection on validation accuracy. Reproduced here rather than imported so
this notebook runs standalone.

In [3]:
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (accuracy_score, recall_score, roc_auc_score,
                             precision_score, f1_score, confusion_matrix)
from tqdm.auto import tqdm

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class TextDS(Dataset):
    def __init__(self, df):
        self.t = df['text'].astype(str).tolist()
        self.y = df['label'].astype(int).tolist()
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], self.y[i]

def loader(df, bs, shuffle):
    return DataLoader(TextDS(df), batch_size=bs, shuffle=shuffle)

def encode(tok, texts, ml):
    return tok(list(texts), max_length=ml, truncation=True,
               padding='max_length', return_tensors='pt').to(DEVICE)

@torch.no_grad()
def predict(model, tok, df, cfg, desc='eval'):
    model.eval(); ys, ps, prs = [], [], []
    for texts, y in tqdm(loader(df, cfg['batch_size'], False), desc=desc, leave=False):
        lg = model(**encode(tok, texts, cfg['max_len'])).logits
        prs.extend(torch.softmax(lg, 1)[:, 1].cpu().numpy())
        ps.extend(torch.argmax(lg, 1).cpu().numpy()); ys.extend(np.asarray(y))
    return np.array(ys), np.array(ps), np.array(prs)

def metrics(y, p, pr):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y, p),
            'precision': precision_score(y, p, zero_division=0),
            'recall': recall_score(y, p, zero_division=0),
            'f1': f1_score(y, p, zero_division=0),
            'roc_auc': roc_auc_score(y, pr) if len(set(y)) > 1 else float('nan'),
            'fpr': fp / (fp + tn) if (fp + tn) else float('nan'),
            'fnr': fn / (fn + tp) if (fn + tp) else float('nan'),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def train_one(hf_id, seed, cfg, tr, va, tag):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'])
    tl = loader(tr, cfg['batch_size'], True)
    best, state = -1.0, None
    for ep in range(cfg['epochs']):
        model.train(); t0, tot = time.time(), 0.0
        for texts, y in tqdm(tl, desc='%s ep%d/%d' % (tag, ep+1, cfg['epochs']),
                             leave=False):
            yt = torch.as_tensor(y).to(DEVICE); opt.zero_grad()
            out = model(**encode(tok, texts, cfg['max_len']), labels=yt)
            out.loss.backward(); opt.step(); tot += out.loss.item()
        yv, pv, _ = predict(model, tok, va, cfg, desc='val')
        vacc = float((yv == pv).mean())
        print('   ep%d loss=%.4f val_acc=%.4f (%.0fs)'
              % (ep+1, tot/max(len(tl), 1), vacc, time.time()-t0), flush=True)
        if vacc > best:
            best = vacc
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if state is not None:
        model.load_state_dict(state); model.to(DEVICE)
    return model, tok, best

def done_keys():
    if not os.path.exists(RUNS):
        return set()
    d = pd.read_csv(RUNS)
    return set(zip(d['model'], d['fold'], d['seed']))

print('core defined')

core defined


## 3. Train --- all 15 folds

Resumable: any (model, fold, seed) already in `results/loto_runs.csv` is
skipped, so the two folds already run (13, 1, seed 42) pick up here
without re-training. Safe to interrupt and re-run.

In [4]:
if STAGE == 1:
    jobs = [(int(r.fold), 42) for r in folds.itertuples()]
else:
    d = pd.read_csv(RUNS); s1 = d[(d.seed == 42) & (d.model == MODEL)]
    assert len(s1) >= len(folds), 'stage 2 needs all stage-1 folds; have %d' % len(s1)
    ext = [int(s1.loc[s1.accuracy.idxmin(), 'fold']),
           int(s1.loc[s1.accuracy.idxmax(), 'fold'])]
    print('extremes from stage 1: folds %s' % ext)
    jobs = [(f, s) for f in ext for s in EXTRA_SEEDS]
print('%d run(s) planned\n' % len(jobs))

for fold, seed in jobs:
    if (MODEL, fold, seed) in done_keys():
        print('[skip] fold %02d seed %d done' % (fold, seed)); continue
    topic = folds.loc[folds.fold == fold, 'held_out_topic'].iloc[0]
    tr = pd.read_csv(os.path.join(LOTO, 'fold_%02d_train.csv' % fold))
    va = pd.read_csv(os.path.join(LOTO, 'fold_%02d_val.csv' % fold))
    te = pd.read_csv(os.path.join(LOTO, 'fold_%02d_test.csv' % fold))
    print('=== fold %02d seed %d  held out: %s ===' % (fold, seed, topic))
    print('    train=%d val=%d test=%d' % (len(tr), len(va), len(te)), flush=True)
    try:
        model, tok, best_val = train_one(HF_ID, seed, CONFIG, tr, va,
                                         'f%02d s%d' % (fold, seed))
    except Exception as ex:
        print('[FAIL] %s: %s' % (type(ex).__name__, ex)); continue

    y, p, pr = predict(model, tok, te, CONFIG, desc='test')
    m = metrics(y, p, pr)
    pd.DataFrame({'uid': te['uid'].values, 'label': y, 'pred': p,
                  'prob_ai': pr}).to_csv(
        os.path.join(PREDS, 'preds_f%02d_%s_seed%d.csv' % (fold, MODEL, seed)),
        index=False)
    pd.DataFrame([{'model': MODEL, 'hf_id': HF_ID, 'fold': fold,
                   'held_out_topic': topic, 'seed': seed, 'n_train': len(tr),
                   'n_val': len(va), 'n_test': len(te),
                   'best_val_accuracy': best_val, **CONFIG, **m}]).to_csv(
        RUNS, mode='a', header=not os.path.exists(RUNS), index=False)
    print('    acc=%.4f recall=%.4f roc=%.4f fpr=%.4f fnr=%.4f'
          % (m['accuracy'], m['recall'], m['roc_auc'], m['fpr'], m['fnr']),
          flush=True)

    del model, tok; gc.collect(); torch.cuda.empty_cache()

print('\nSTAGE %d COMPLETE' % STAGE)

15 run(s) planned

=== fold 00 seed 42  held out: Distance learning ===
    train=23966 val=4230 test=4314


f00 s42 ep1/3:   0%|          | 0/2996 [00:00<?, ?it/s]

val:   0%|          | 0/529 [00:00<?, ?it/s]

   ep1 loss=0.0505 val_acc=0.9267 (574s)


f00 s42 ep2/3:   0%|          | 0/2996 [00:00<?, ?it/s]

...

f14 s42 ep3/3:   0%|          | 0/3652 [00:00<?, ?it/s]

val:   0%|          | 0/645 [00:00<?, ?it/s]

   ep3 loss=0.0119 val_acc=0.9971 (695s)


test:   0%|          | 0/78 [00:00<?, ?it/s]

    acc=0.9935 recall=0.9968 roc=0.9999 fpr=0.0097 fnr=0.0032

STAGE 1 COMPLETE


## 4. RQ1 --- RoBERTa vs. the controls, all 15 folds

Same reporting shape as the main notebook's section 5, filtered to
`model=='roberta'`, with DeBERTa's numbers alongside for the comparison
the paper's limitation calls for.

In [5]:
runs = pd.read_csv(RUNS)
base = pd.read_csv(BASE) if os.path.exists(BASE) else pd.DataFrame()

rob = runs[(runs.model == 'roberta') & (runs.seed == 42)]
deb = runs[(runs.model == 'deberta-v3') & (runs.seed == 42)]
piv = base.pivot_table(index='fold', columns='baseline', values='accuracy')

if len(rob) < len(folds):
    print('RoBERTa folds done so far: %d / %d' % (len(rob), len(folds)))
    print(sorted(rob.fold.unique()))
else:
    print('all %d folds complete for RoBERTa\n' % len(folds))

cmp_ = rob[['fold', 'held_out_topic', 'accuracy', 'recall']].merge(
    deb[['fold', 'accuracy', 'recall']], on='fold', suffixes=('_roberta', '_deberta')
).merge(piv, on='fold', how='left').sort_values('fold')

for c in ['accuracy_roberta', 'accuracy_deberta', 'recall_roberta',
         'recall_deberta', 'length_only', 'tfidf']:
    if c in cmp_:
        cmp_[c] = (cmp_[c] * 100).round(2)
cmp_['roberta_vs_tfidf'] = (cmp_.accuracy_roberta - cmp_.tfidf).round(2)
cmp_['deberta_vs_tfidf'] = (cmp_.accuracy_deberta - cmp_.tfidf).round(2)
cmp_['roberta_vs_deberta'] = (cmp_.accuracy_roberta - cmp_.accuracy_deberta).round(2)

print(cmp_[['fold', 'held_out_topic', 'accuracy_roberta', 'accuracy_deberta',
           'roberta_vs_deberta', 'tfidf', 'roberta_vs_tfidf',
           'deberta_vs_tfidf']].to_string(index=False))

if len(cmp_) == len(folds):
    print()
    print('RoBERTa: mean acc %.2f  wins vs TF-IDF %d/%d  mean margin %+.2f'
          % (cmp_.accuracy_roberta.mean(), (cmp_.roberta_vs_tfidf > 0).sum(),
             len(cmp_), cmp_.roberta_vs_tfidf.mean()))
    print('DeBERTa: mean acc %.2f  wins vs TF-IDF %d/%d  mean margin %+.2f'
          % (cmp_.accuracy_deberta.mean(), (cmp_.deberta_vs_tfidf > 0).sum(),
             len(cmp_), cmp_.deberta_vs_tfidf.mean()))

all 15 folds complete for RoBERTa

 fold                        held_out_topic  accuracy_roberta  accuracy_deberta  roberta_vs_deberta  tfidf  roberta_vs_tfidf  deberta_vs_tfidf
    0                     Distance learning             96.64             98.79               -2.15  99.44             -2.80             -0.65
    1                       Car-free cities             99.59             99.61               -0.02  99.32              0.27              0.29
    2      Does the electoral college work?             90.06             99.48               -9.42  98.95             -8.89              0.53
    3             Seeking multiple opinions             98.16             96.01                2.15  98.20             -0.04             -2.19
    4  Mandatory extracurricular activities             98.12             98.36               -0.24  99.00             -0.88             -0.64
    5                       Summer projects             97.21             97.79               -0.58  99.16 

## 5. RQ2 --- who absorbs RoBERTa's false accusations

Same join as the main notebook's section 6, scoped to RoBERTa's
predictions, so the grade/ELL breakdown can be compared architecture to
architecture once all 15 folds are in.

In [6]:
import glob

MIN_N = 30
pers = pd.read_csv(PERSUADE, low_memory=False)
pers['uid'] = pers['full_text'].map(h)
pmeta = pers.drop_duplicates('uid').set_index('uid')

fs = sorted(glob.glob(os.path.join(PREDS, 'preds_f*_roberta_seed42.csv')))
if not fs:
    print('no RoBERTa predictions yet')
else:
    P = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
    auth = P[P.label == 0].join(pmeta[['holistic_essay_score', 'ell_status']],
                                on='uid', how='inner')
    print('authentic essays with writer info: %d of %d (%.0f%%), from %d fold(s)'
          % (len(auth), (P.label == 0).sum(),
             100*len(auth)/max((P.label == 0).sum(), 1), len(fs)))

    print('\n--- by holistic grade ---')
    g = auth.groupby('holistic_essay_score')['pred'].agg(['size', 'mean'])
    g['mean'] = (g['mean']*100).round(2)
    print(g.rename(columns={'size': 'n', 'mean': 'FPR_pct'}).to_string())

    print('\n--- by English-language status ---')
    g = auth.groupby('ell_status')['pred'].agg(['size', 'mean'])
    g['mean'] = (g['mean']*100).round(2)
    print(g.rename(columns={'size': 'n', 'mean': 'FPR_pct'}).to_string())

authentic essays with writer info: 13202 of 14183 (93%), from 15 fold(s)

--- by holistic grade ---
                         n  FPR_pct
holistic_essay_score               
1                      434     0.00
2                     2314     0.39
3                     3805     0.87
4                     3846     2.68
5                     2166     4.94
6                      637    11.93

--- by English-language status ---
                n  FPR_pct
ell_status                
               20     0.00
No          11483     2.76
Yes          1271     0.79
